# Can a Language Model Pick Your Features?

**LLM-guided feature selection on the Adult Census Income dataset, compared against statistics and a trained model**

Feature selection is the step where you decide which columns actually deserve a place in your model. Too many features and the model chases noise; too few and it misses the signal. The usual tools are statistical: measure how strongly each column relates to the target, or train a model and ask which inputs it leans on.

This notebook tries a third option. We hand a transformer language model nothing but the **names and plain-English descriptions** of the columns, and ask it to score how useful each one should be. It never sees a single row of data. Then we put its picks up against two data-driven methods and see whose choices build the better model.

The idea comes from [LLM-Select (Jeong, Lipton, and Ravikumar, 2024)](https://arxiv.org/abs/2407.02694), which found that large language models can rank features competitively with classic methods using only domain knowledge.

| Section | What happens |
| --- | --- |
| 1. The data | Load 32,561 census records from Hugging Face and explore them |
| 2. A baseline model | Train a gradient-boosted model on every feature |
| 3. Statistical ranking | Mutual information between each feature and income |
| 4. Model-based ranking | Permutation importance from the trained model |
| 5. Language-model ranking | Ask a transformer LLM to score the features from their descriptions alone |
| 6. Comparing the rankings | Where the three methods agree and disagree |
| 7. Does it matter? | Retrain with the top *k* features from each method |
| 8. Takeaways | What this means for real projects |

> **Running it yourself:** everything except Section 5 runs anywhere. Section 5 calls any OpenAI-compatible endpoint (llama.cpp, Ollama, vLLM, or a hosted API). If none is available, the notebook falls back to the saved scores in `llm_feature_scores.json`, so every cell still runs.

## Setup

The libraries are the standard Python data stack plus two extras: `datasets` to pull the data from Hugging Face, and `openai`, which is only used as a client for whichever language model you point it at.

```bash
pip install pandas scikit-learn plotly datasets openai scipy
```

The model settings come from environment variables so nothing secret lives in the notebook. The defaults point at a local [llama.cpp](https://github.com/ggml-org/llama.cpp) server running **MiniCPM-V 4.7 35B-A3B**, an open-weights mixture-of-experts transformer. To use **Mistral NeMo** instead, run `ollama pull mistral-nemo` and set `LLM_BASE_URL=http://localhost:11434/v1` and `LLM_MODEL=mistral-nemo`.

In [1]:
import json
import os
import re
import warnings
from pathlib import Path

# keep the output to results: no download progress bars or library notices
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from datasets import load_dataset
from huggingface_hub.utils import logging as hf_logging
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

hf_logging.set_verbosity_error()
RANDOM_STATE = 42

# language model settings (any OpenAI-compatible server)
LLM_BASE_URL = os.environ.get("LLM_BASE_URL", "http://127.0.0.1:8080/v1")
LLM_API_KEY = os.environ.get("LLM_API_KEY", "not-needed")
LLM_MODEL = os.environ.get("LLM_MODEL", "minicpm-v")
LLM_CACHE = Path("llm_feature_scores.json")
# reuse the saved scores so results are reproducible; set True to score again with your own model
REFRESH_LLM = os.environ.get("REFRESH_LLM") == "1"

# one look for every chart: dark, transparent, and colorblind-friendly
COLORS = {"Mutual information": "#38bdf8", "Permutation importance": "#a78bfa", "Language model": "#34d399", "Random": "#94a3b8"}
pio.templates["post"] = go.layout.Template(
    layout=dict(
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        font=dict(family="Inter, system-ui, sans-serif", color="#cbd5e1", size=13),
        title=dict(font=dict(size=17, color="#f1f5f9"), x=0, xanchor="left"),
        xaxis=dict(gridcolor="rgba(148,163,184,0.15)", zerolinecolor="rgba(148,163,184,0.3)"),
        yaxis=dict(gridcolor="rgba(148,163,184,0.15)", zerolinecolor="rgba(148,163,184,0.3)"),
        colorway=list(COLORS.values()),
        margin=dict(l=10, r=10, t=60, b=10),
        legend=dict(bgcolor="rgba(0,0,0,0)"),
    )
)
pio.templates.default = "plotly_dark+post"
# show charts in JupyterLab and VS Code, and keep their data in the saved notebook
pio.renderers.default = "plotly_mimetype+notebook_connected"

## 1. The data

The [Adult Census Income](https://huggingface.co/datasets/scikit-learn/adult-census-income) dataset comes from the 1994 U.S. Census. Each row is one person, and the question is simple: **does this person earn more than $50,000 a year?** It is one of the most widely used datasets in machine learning, which makes it a fair test, and it has a few traps worth noticing.

| Feature | What it means |
| --- | --- |
| `age` | Age in years |
| `workclass` | Employer type: private, self-employed, government, and so on |
| `fnlwgt` | Census sampling weight: how many people in the population this row represents |
| `education` | Highest level of education, as a label |
| `education.num` | The same education level, as a number from 1 to 16 |
| `marital.status` | Married, never married, divorced, and so on |
| `occupation` | Broad job category |
| `relationship` | Role in the household: husband, wife, own child, and so on |
| `race` | Race as recorded by the census |
| `sex` | Sex as recorded by the census |
| `capital.gain` | Income from investment gains |
| `capital.loss` | Losses from investments |
| `hours.per.week` | Usual hours worked per week |
| `native.country` | Country of origin |

Two of those deserve a flag before we start. **`fnlwgt` is not a fact about the person at all**; it is a weight the census uses to scale the sample up to the population. And **`education` and `education.num` carry identical information** in two formats. A good feature selection method should treat the first as noise and should not need both of the second.

In [2]:
raw = load_dataset("scikit-learn/adult-census-income", split="train").to_pandas()

df = raw.replace("?", np.nan)
CATEGORICAL = ["workclass", "education", "marital.status", "occupation", "relationship", "race", "sex", "native.country"]
NUMERIC = ["age", "fnlwgt", "education.num", "capital.gain", "capital.loss", "hours.per.week"]
FEATURES = [f for f in raw.columns if f != "income"]

for col in CATEGORICAL:
    df[col] = df[col].fillna("Unknown").astype("category")
y = (df["income"] == ">50K").astype(int)
X = df[FEATURES]

print(f"{len(df):,} people, {len(FEATURES)} features")
print(f"{y.mean():.1%} earn more than $50K")
X.head()

32,561 people, 14 features
24.1% earn more than $50K


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country
0,90,Unknown,77053,HS-grad,9,Widowed,Unknown,Not-in-family,White,Female,0,4356,40,United-States
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States
2,66,Unknown,186061,Some-college,10,Widowed,Unknown,Unmarried,Black,Female,0,4356,40,United-States
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States


About one in four people in the data earns more than $50K, so the classes are imbalanced. That is why the models below are scored with **ROC AUC** instead of plain accuracy: a model that always guessed "no" would be 76% accurate and completely useless, but its AUC would be 0.5, the same as a coin flip.

Before ranking anything, a quick look at how a few features relate to income.

In [3]:
# count ages into 2-year bins per group, so the chart carries 70 numbers instead of 32,000 rows
bins = np.arange(16, 92, 2)
age_mix = pd.concat([
    pd.DataFrame({"age": bins[:-1], "upper": bins[:-1] + 1, "share": np.histogram(df.loc[y == flag, "age"], bins=bins)[0] / (y == flag).sum(), "Income": label})
    for flag, label in [(0, "$50K or less"), (1, "More than $50K")]
])
age_fig = px.bar(
    age_mix, x="age", y="share", color="Income", barmode="overlay", opacity=0.75, custom_data=["upper"],
    color_discrete_sequence=["#94a3b8", "#34d399"],
    title="Higher earners skew older, peaking in their 40s and 50s",
    labels={"age": "Age", "share": "Share of group"},
)
age_fig.update_traces(hovertemplate="Ages %{x} to %{customdata[0]}: %{y:.1%} of the group<extra></extra>")
age_fig.update_layout(yaxis_tickformat=".0%", legend_title_text="", bargap=0.05)
age_fig.show()

In [4]:
edu = (
    df.assign(high=y)
    .groupby("education", observed=True)
    .agg(rate=("high", "mean"), people=("high", "size"), level=("education.num", "first"))
    .sort_values("level")
    .reset_index()
)
edu_fig = px.bar(
    edu, x="education", y="rate", custom_data=["people"],
    title="The chance of earning over $50K climbs with every step of education",
    labels={"education": "", "rate": "Share earning over $50K"},
    color_discrete_sequence=["#34d399"],
)
edu_fig.update_traces(hovertemplate="%{x}<br>%{y:.0%} earn over $50K<br>%{customdata[0]:,} people<extra></extra>")
edu_fig.update_layout(yaxis_tickformat=".0%", xaxis_tickangle=-40)
edu_fig.show()

## 2. A baseline model

To judge any feature ranking we need a model to test it with. A **histogram gradient boosting** classifier is a strong, fast default for tabular data, and it handles categorical columns natively, so no one-hot encoding is needed.

We hold out 25% of the people as a test set the model never trains on, and score it with ROC AUC.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)


def make_model():
    return HistGradientBoostingClassifier(categorical_features="from_dtype", random_state=RANDOM_STATE)


baseline = make_model().fit(X_train, y_train)
baseline_auc = roc_auc_score(y_test, baseline.predict_proba(X_test)[:, 1])
print(f"Test ROC AUC with all {len(FEATURES)} features: {baseline_auc:.3f}")

Test ROC AUC with all 14 features: 0.922


That is the number to beat, or at least to match with fewer features. A simpler model that is nearly as accurate is usually the better model: cheaper to run, easier to explain, and less likely to break when the data shifts.

## 3. Statistical ranking: mutual information

**Mutual information** measures how much knowing a feature reduces uncertainty about the target. Zero means the feature tells you nothing about income; higher means it tells you more. It captures non-linear relationships, but it scores each feature on its own, so it cannot see that two features are redundant.

In [6]:
X_codes = X_train.copy()
for col in CATEGORICAL:
    X_codes[col] = X_codes[col].cat.codes

mi = pd.Series(
    mutual_info_classif(X_codes, y_train, discrete_features=[c in CATEGORICAL for c in FEATURES], random_state=RANDOM_STATE),
    index=FEATURES,
).sort_values()

mi_fig = px.bar(
    x=mi.values, y=mi.index, orientation="h",
    title="Mutual information: relationship and marital status lead",
    labels={"x": "Mutual information with income", "y": ""},
    color_discrete_sequence=[COLORS["Mutual information"]],
)
mi_fig.update_traces(hovertemplate="%{y}: %{x:.3f}<extra></extra>")
mi_fig.show()

## 4. Model-based ranking: permutation importance

**Permutation importance** asks the trained model directly. For each feature, it shuffles that column in the test set, breaking its link to income, and measures how much the model's AUC drops. A big drop means the model relied on that feature. We repeat the shuffle ten times per feature to get error bars.

Unlike mutual information, this accounts for redundancy: if two features carry the same signal, shuffling one barely hurts because the model can lean on the other.

In [7]:
perm = permutation_importance(
    baseline, X_test, y_test, scoring="roc_auc", n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1
)
pi = pd.DataFrame({"mean": perm.importances_mean, "std": perm.importances_std}, index=FEATURES).sort_values("mean")

pi_fig = px.bar(
    pi, x="mean", y=pi.index, orientation="h", error_x="std",
    title="Permutation importance: what the trained model actually relies on",
    labels={"mean": "Drop in ROC AUC when shuffled", "index": ""},
    color_discrete_sequence=[COLORS["Permutation importance"]],
)
pi_fig.update_traces(hovertemplate="%{y}: %{x:.4f}<extra></extra>")
pi_fig.update_layout(yaxis_title="")
pi_fig.show()

## 5. Language-model ranking

Now the experiment. We give a transformer language model the same table of feature descriptions a person would read, explain the prediction task, and ask it to score each feature from 0 to 1 for how useful it should be, along with a one-sentence reason.

The model gets **no data**: no rows, no statistics, no correlations. Everything it knows comes from what it learned in pre-training about how income relates to age, education, work, and so on. That matters in practice: this kind of ranking works before you have collected any data at all, and it never exposes sensitive records to the model.

Language models are not deterministic, so we ask five times and average the scores. The spread between runs becomes an error bar.

The scores from the run used in the blog post are saved in `llm_feature_scores.json` and reused by default, so your results match the post. Set `REFRESH_LLM = True` (or the environment variable `REFRESH_LLM=1`) to score the features again with your own model.

In [8]:
DESCRIPTIONS = {
    "age": "Age in years",
    "workclass": "Employer type: private, self-employed, government, and so on",
    "fnlwgt": "Census sampling weight: how many people in the population this row represents",
    "education": "Highest level of education, as a label",
    "education.num": "The same education level, as a number from 1 to 16",
    "marital.status": "Married, never married, divorced, and so on",
    "occupation": "Broad job category",
    "relationship": "Role in the household: husband, wife, own child, and so on",
    "race": "Race as recorded by the census",
    "sex": "Sex as recorded by the census",
    "capital.gain": "Income from investment gains",
    "capital.loss": "Losses from investments",
    "hours.per.week": "Usual hours worked per week",
    "native.country": "Country of origin",
}

PROMPT = (
    "You are helping select features for a model that predicts whether a person's annual income is above $50,000, "
    "using records from the 1994 U.S. Census.\n\n"
    "Features:\n" + "\n".join(f"- {name}: {desc}" for name, desc in DESCRIPTIONS.items()) + "\n\n"
    "For each feature, give an importance score from 0 (useless for predicting income) to 1 (extremely useful), "
    "based on your knowledge of what drives income. Score features independently; several can share a score.\n"
    'Reply with only a JSON object mapping each feature name to {"score": number, "reason": "one short sentence"}.'
)
print(PROMPT)

You are helping select features for a model that predicts whether a person's annual income is above $50,000, using records from the 1994 U.S. Census.

Features:
- age: Age in years
- workclass: Employer type: private, self-employed, government, and so on
- fnlwgt: Census sampling weight: how many people in the population this row represents
- education: Highest level of education, as a label
- education.num: The same education level, as a number from 1 to 16
- marital.status: Married, never married, divorced, and so on
- occupation: Broad job category
- relationship: Role in the household: husband, wife, own child, and so on
- race: Race as recorded by the census
- sex: Sex as recorded by the census
- capital.gain: Income from investment gains
- capital.loss: Losses from investments
- hours.per.week: Usual hours worked per week
- native.country: Country of origin

For each feature, give an importance score from 0 (useless for predicting income) to 1 (extremely useful), based on your kn

In [9]:
N_RUNS = 5


def ask_llm(n_runs=N_RUNS):
    from openai import OpenAI

    client = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY, timeout=180)
    runs = []
    for i in range(n_runs):
        reply = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": PROMPT}],
            temperature=0.7,
            max_tokens=1500,
        ).choices[0].message.content
        parsed = json.loads(re.search(r"\{.*\}", reply, re.S).group(0))  # tolerate text around the JSON
        runs.append({f: parsed[f] for f in FEATURES})
        print(f"run {i + 1}/{n_runs} done")
    return runs


CACHE_URL = "https://raw.githubusercontent.com/towardsuffering/rotunno.io/main/src/data/blog/en/llm-feature-selection/llm_feature_scores.json"


def load_cache():
    # opened from GitHub in Colab, the saved scores aren't next to the notebook, so fetch them
    if not LLM_CACHE.exists():
        import urllib.request

        LLM_CACHE.write_bytes(urllib.request.urlopen(CACHE_URL).read())
    return json.loads(LLM_CACHE.read_text())


if REFRESH_LLM or not (LLM_CACHE.exists() or os.environ.get("COLAB_RELEASE_TAG")):
    llm_runs = ask_llm()
    LLM_CACHE.write_text(json.dumps({"model": LLM_MODEL, "prompt": PROMPT, "runs": llm_runs}, indent=1))
    llm_model_used = LLM_MODEL
    print(f"Scored by {LLM_MODEL}; saved to {LLM_CACHE}")
else:
    cached = load_cache()
    llm_runs, llm_model_used = cached["runs"], cached["model"]
    print(f"Using saved scores from {llm_model_used} ({len(llm_runs)} runs)")

Using saved scores from minicpm-v (5 runs)


In [10]:
scores = pd.DataFrame([{f: float(run[f]["score"]) for f in FEATURES} for run in llm_runs])
llm = pd.DataFrame({"mean": scores.mean(), "std": scores.std()}).sort_values("mean")
reasons = {f: llm_runs[0][f]["reason"] for f in FEATURES}

llm_fig = px.bar(
    llm, x="mean", y=llm.index, orientation="h", error_x="std",
    custom_data=[[reasons[f] for f in llm.index]],
    title=f"Language model scores ({llm_model_used}, average of {len(llm_runs)} runs)",
    labels={"mean": "Importance score (0 to 1)", "index": ""},
    color_discrete_sequence=[COLORS["Language model"]],
)
llm_fig.update_traces(hovertemplate="<b>%{y}</b>: %{x:.2f}<br>%{customdata[0]}<extra></extra>")
llm_fig.update_layout(yaxis_title="", xaxis_range=[0, 1.05])
llm_fig.show()

pd.DataFrame({"score": llm["mean"].round(2), "reason": [reasons[f] for f in llm.index]}, index=llm.index).iloc[::-1]

,score,reason
occupation,0.84,The specific job category is one of the strong...
education.num,0.80,Provides the same predictive information as ed...
education,0.80,Higher education levels are strongly correlate...
age,0.70,Income generally increases with experience and...
capital.gain,0.68,Significant investment gains often indicate we...
hours.per.week,0.60,Working more hours generally leads to higher i...
workclass,0.60,Employer type often correlates with industry s...
sex,0.50,"Gender pay gaps exist, making it a weak but st..."
race,0.44,"While systemic disparities exist, it is a sens..."
capital.loss,0.40,Investment losses can indicate active investin...


Hover over any bar to read the model's reasoning. Two things to check against the warnings from Section 1: what score did it give `fnlwgt`, the sampling weight that says nothing about a person, and did it notice that `education` and `education.num` are the same information?

## 6. Comparing the rankings

Each method produced a score on its own scale, so the fair comparison is by **rank**: 1 for the feature a method thinks matters most, 14 for the least. The chart below follows every feature across the three methods. Flat lines mean agreement; steep lines mean the methods see that feature very differently.

In [11]:
rankings = pd.DataFrame({
    "Mutual information": mi.rank(ascending=False),
    "Permutation importance": pi["mean"].rank(ascending=False),
    "Language model": llm["mean"].rank(ascending=False, method="first"),
}).loc[FEATURES]

methods = list(rankings.columns)
bump = go.Figure()
for feature, row in rankings.iterrows():
    highlight = feature in ("fnlwgt", "education", "education.num", "capital.gain")
    bump.add_trace(go.Scatter(
        x=methods, y=row.values, mode="lines+markers+text", name=feature,
        text=["", "", feature], textposition="middle right",
        line=dict(width=3 if highlight else 1.5),
        opacity=1 if highlight else 0.55,
        hovertemplate=f"<b>{feature}</b><br>%{{x}}: rank %{{y:.0f}}<extra></extra>",
    ))
bump.update_layout(
    title="How each feature ranks under each method (1 = most important)",
    yaxis=dict(autorange="reversed", title="Rank", dtick=1),
    xaxis=dict(title=""), showlegend=False, height=560, margin=dict(r=120),
)
bump.show()

In [12]:
corr = rankings.corr(method="spearman")
corr_fig = px.imshow(
    corr, text_auto=".2f", zmin=-1, zmax=1, color_continuous_scale="Tealgrn",
    title="Rank agreement between methods (Spearman correlation)",
)
corr_fig.update_layout(coloraxis_showscale=False, height=420)
corr_fig.show()

A Spearman correlation of 1 would mean two methods order the features identically; 0 would mean no relationship at all. The interesting question is which pair agrees most, and whether the language model, working from descriptions alone, lands closer to the statistics or to the trained model.

## 7. Does it matter? Retraining with the top *k* features

Agreement is nice, but the real test is whether a ranking builds good models. For each method we take its top 1 feature, then its top 2, and so on up to all 14, and measure the cross-validated ROC AUC at each step. A **random** ordering, averaged over five shuffles, shows what no method at all looks like.

A good ranking climbs fast: it reaches near-baseline accuracy with only a handful of features.

In [13]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)


def auc_with(features):
    return cross_val_score(make_model(), X_train[features], y_train, cv=cv, scoring="roc_auc").mean()


orders = {method: list(rankings[method].sort_values().index) for method in methods}
rng = np.random.default_rng(RANDOM_STATE)
random_orders = [list(rng.permutation(FEATURES)) for _ in range(5)]

ks = range(1, len(FEATURES) + 1)
curves = {method: [auc_with(order[:k]) for k in ks] for method, order in orders.items()}
curves["Random"] = [np.mean([auc_with(order[:k]) for order in random_orders]) for k in ks]

curve_df = pd.DataFrame(curves, index=list(ks))
curve_fig = go.Figure()
for method in curve_df.columns:
    curve_fig.add_trace(go.Scatter(
        x=curve_df.index, y=curve_df[method], mode="lines+markers", name=method,
        line=dict(color=COLORS[method], width=3 if method == "Language model" else 2, dash="dot" if method == "Random" else "solid"),
        hovertemplate=f"{method}<br>top %{{x}} features: AUC %{{y:.3f}}<extra></extra>",
    ))
curve_fig.add_hline(y=curve_df.iloc[-1, 0], line_dash="dash", line_color="rgba(148,163,184,0.5)", annotation_text="all features")
curve_fig.update_layout(
    title="Model accuracy using only each method's top k features",
    xaxis=dict(title="Number of features used (k)", dtick=1),
    yaxis=dict(title="Cross-validated ROC AUC"),
    legend=dict(orientation="h", y=-0.2), height=500,
)
curve_fig.show()
curve_df.round(3).T

,1,2,3,4,5,6,7,8,9,10,11,12,13,14
Mutual information,0.782,0.784,0.843,0.872,0.908,0.914,0.914,0.918,0.926,0.925,0.926,0.926,0.927,0.926
Permutation importance,0.710,0.780,0.868,0.872,0.903,0.913,0.922,0.926,0.927,0.927,0.927,0.927,0.927,0.926
Language model,0.730,0.768,0.768,0.823,0.862,0.864,0.873,0.884,0.884,0.919,0.926,0.927,0.928,0.926
Random,0.571,0.691,0.786,0.830,0.870,0.880,0.886,0.908,0.909,0.916,0.922,0.925,0.926,0.926


In [14]:
target = curve_df.iloc[-1, 0] - 0.01
needed = {m: int(curve_df.index[curve_df[m] >= target][0]) for m in curve_df.columns}
print(f"Features needed to get within 0.01 AUC of the full model ({curve_df.iloc[-1, 0]:.3f}):")
for method, k in sorted(needed.items(), key=lambda kv: kv[1]):
    print(f"  {method:<24} {k:>2} features: {', '.join(orders[method][:k]) if method in orders else '(random)'}")

Features needed to get within 0.01 AUC of the full model (0.926):
  Permutation importance    7 features: age, capital.gain, marital.status, relationship, occupation, education, capital.loss
  Mutual information        8 features: relationship, marital.status, capital.gain, age, education.num, occupation, education, hours.per.week
  Language model           10 features: occupation, education, education.num, age, capital.gain, workclass, hours.per.week, sex, race, marital.status
  Random                   11 features: (random)


## 8. Takeaways

These numbers come from the run saved with this notebook (MiniCPM-V 4.7, five runs averaged). With `REFRESH_LLM` on, a different model will tell a somewhat different story, which is part of the point.

**The language model got the judgment calls right.** It gave `fnlwgt` a score of 0.06, explaining that it is "a sampling weight for population estimation, not a predictor of individual income," and it did so in all five runs. Mutual information, working from the data alone, ranked that same meaningless column 10th of 14, ahead of four real features. Knowing what a column *is* turns out to be something only the language model could do.

**But it missed what the data knew.** The two strongest signals by both data-driven methods were `relationship` and `marital.status`, which in 1994 census data act as a stand-in for "married, two-income household." The language model ranked them 12th and 10th. It also noticed that `education.num` duplicates `education`, said so in its own reasoning, and then scored both 0.80 anyway, spending its second and third picks on the same information. That is why its curve goes flat from two features to three.

**The scoreboard.** To get within 0.01 of the full model's cross-validated AUC (0.926):

| Method | Features needed |
| --- | --- |
| Permutation importance | 7 |
| Mutual information | 8 |
| Language model | 10 |
| Random order | 11 |

The two data-driven methods mostly agreed with each other (Spearman 0.80), while the language model agreed only weakly with either (0.27 and 0.33). It beat random ordering only with its first two picks, then trailed it from three features to ten.

**So: let the language model propose, and let the data decide.** A language model is genuinely useful before you have data, for deciding what to collect, and for catching columns that are artifacts of how the data was gathered, like sampling weights, IDs, or timestamps. It is not a replacement for measuring what the data actually says. The strongest workflow uses both: the model's knowledge to frame the problem, and permutation importance plus a top-*k* curve like the one in Section 7 to make the final call.

### Caveats

- **This dataset is famous.** It appears in countless tutorials, so the language model may partly remember this dataset rather than reasoning about it. A fairer test would use a private or newly collected dataset.
- **Scores are not probabilities.** The 0 to 1 scale is the model's judgment, and different models or prompts can produce different rankings. Averaging several runs helps.
- **Sensitive attributes need human judgment.** Whether `race` or `sex` belong in an income model is an ethical and legal question, not a feature selection one.

### Try it yourself

- Set `REFRESH_LLM = True`, point `LLM_MODEL` and `LLM_BASE_URL` at a different model, and see whether its ranking changes.
- Rewrite the prompt to warn about redundant features and check whether `education.num` drops.
- Point the notebook at another Hugging Face tabular dataset by replacing the dataset name and the `DESCRIPTIONS` table.